In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")

print("Path to dataset files:", path)

/Users/flaviafuscaldi/Desktop/fl_thesis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 778M/778M [01:39<00:00, 8.21MB/s] 

Extracting files...


Path to dataset files: /Users/flaviafuscaldi/.cache/kagglehub/datasets/tawsifurrahman/covid19-radiography-database/versions/5


In [ ]:
import shutil
import os

source_path = path
destination_path = "/Users/flavia/Desktop/fl_thesis"

shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print("Dataset moved successfully.")


Dataset moved successfully.


In [3]:
from pathlib import Path

dataset_path = Path("COVID-19_Radiography_Dataset")

print("Exists:", dataset_path.exists())
print("Subfolders:", [p.name for p in dataset_path.iterdir()])


Exists: True
Subfolders: ['Lung_Opacity.metadata.xlsx', 'Viral Pneumonia.metadata.xlsx', 'Viral Pneumonia', 'COVID.metadata.xlsx', 'Normal.metadata.xlsx', 'Lung_Opacity', 'Normal', 'COVID', 'README.md.txt']


In [15]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(
    root=str(dataset_path),
    transform=transform
)

print("Classes:", dataset.classes)
print("Total samples:", len(dataset))


Classes: ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Total samples: 42330


In [16]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

from aijack.collaborative.fedavg import FedAVGClient, FedAVGServer, FedAVGAPI


def evaluate_global_model(dataloader, client_id=-1):
    def _evaluate_global_model(api):
        test_loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(api.device), target.to(api.device)
                if client_id == -1:
                    output = api.server(data)
                else:
                    output = api.clients[client_id](data)
                test_loss += F.cross_entropy(output, target, reduction="sum").item() 
                pred = output.argmax(
                    dim=1, keepdim=True
                )  
                correct += pred.eq(target.view_as(pred)).sum().item()

        test_loss /= len(dataloader.dataset)
        accuracy = 100.0 * correct / len(dataloader.dataset)
        print(f"Test set: Average loss: {test_loss}, Accuracy: {accuracy}")

    return _evaluate_global_model



In [17]:
training_batch_size = 64
test_batch_size = 64
num_rounds = 5
lr = 0.001
seed = 0
client_size = 2
criterion = F.cross_entropy

In [18]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [19]:
from pathlib import Path
from torch.utils.data import DataLoader, Subset

def prepare_dataloader(num_clients, myid, train=True, path="COVID-19_Radiography_Dataset", seed=0):
    """
    num_clients: total number of clients
    myid: client id in [1..num_clients]  (AIJack tutorial style)
    train: True => client train loader, False => global test loader
    path: folder containing class subfolders (COVID, Normal, etc.)
    """
    fix_seed(seed)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = datasets.ImageFolder(root=str(Path(path)), transform=transform)

    # create a fixed train/test split (80/20) so every client uses the same split
    idxs = list(range(len(dataset)))
    random.shuffle(idxs)
    split = int(0.8 * len(idxs))
    train_idxs = idxs[:split]
    test_idxs  = idxs[split:]

    if train:
        # split train among clients
        client_splits = np.array_split(train_idxs, num_clients)
        client_idx = client_splits[myid - 1]   # myid is 1-based
        subset = Subset(dataset, client_idx)

        train_loader = DataLoader(
            subset,
            batch_size=training_batch_size,
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )
        return train_loader
    else:
        # global test loader (same for everyone)
        subset = Subset(dataset, test_idxs)
        test_loader = DataLoader(
            subset,
            batch_size=test_batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )
        return test_loader


In [20]:
class Net(nn.Module):
    def __init__(self, num_classes=4):
        super(Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 112x112
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 56x56
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 28x28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x  # logits (✅ for cross_entropy)


In [21]:
fix_seed(seed)

local_dataloaders = [
    prepare_dataloader(client_size, myid=c+1, train=True, seed=seed)
    for c in range(client_size)
]

test_dataloader = prepare_dataloader(client_size, myid=1, train=False, seed=seed)

x, y = next(iter(train0))
print("Batch X:", x.shape, "Batch y:", y.shape)
print("Unique labels in batch:", torch.unique(y))


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x10cd9af80>
Traceback (most recent call last):
  File "/Users/flaviafuscaldi/Desktop/fl_thesis/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/Users/flaviafuscaldi/Desktop/fl_thesis/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1618, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/homebrew/Cellar/python@3.10/3.10.19_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/opt/homebrew/Cellar/python@3.10/3.10.19_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/opt/homebrew/Cellar/python@3.10/3.10.19_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocess

Batch X: torch.Size([64, 3, 224, 224]) Batch y: torch.Size([64])
Unique labels in batch: tensor([0, 1, 2, 3])


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fix_seed(seed)

In [ ]:
local_dataloaders = [prepare_dataloader(client_size, myid=c+1, train=True, seed=seed)
                     for c in range(client_size)]

In [14]:
clients = [FedAVGClient(Net().to(device), user_id=c) for c in range(client_size)]
local_optimizers = [optim.SGD(client.parameters(), lr=lr) for client in clients]

server = FedAVGServer(clients, Net().to(device))

api = FedAVGAPI(
    server,
    clients,
    criterion,
    local_optimizers,
    local_dataloaders,
    num_communication=num_rounds,
    custom_action=evaluate_global_model(test_dataloader),
)
api.run()

KeyboardInterrupt: 

In [3]:
# =========================
# AIJack FedAvg on COVID-19 Radiography Dataset (CPU-optimized, copy-paste)
# - Uses CrossEntropyLoss (logits)
# - Resize to 128x128 for speed on CPU
# - num_workers=0 to avoid dataloader stalls in VSCode/Jupyter
# =========================

from pathlib import Path
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from aijack.collaborative.fedavg import FedAVGClient, FedAVGServer, FedAVGAPI


# -------------------------
# 0) Hyperparameters
# -------------------------
training_batch_size = 64
test_batch_size = 64
num_rounds = 5
lr = 0.001
seed = 0
client_size = 2
criterion = F.cross_entropy  # using CrossEntropy (logits)

# CPU speed knobs (optional)
IMG_SIZE = 128               # was 224; 128 is much faster on CPU
NUM_WORKERS = 0              # safest for notebooks
PIN_MEMORY = False


# -------------------------
# 1) Reproducibility
# -------------------------
def fix_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -------------------------
# 2) Dataset path check (local folder)
# -------------------------
dataset_path = Path("COVID-19_Radiography_Dataset")
print("Exists:", dataset_path.exists())
print("Subfolders:", [p.name for p in dataset_path.iterdir()])


# -------------------------
# 3) Evaluation callback (prints after each round)
# -------------------------
def evaluate_global_model(dataloader, client_id: int = -1):
    """
    If client_id == -1 => evaluate server/global model
    else => evaluate a specific client model (rarely needed)
    """
    def _evaluate_global_model(api):
        test_loss = 0.0
        correct = 0

        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(api.device), target.to(api.device)

                # Depending on AIJack version, server might not be callable.
                if client_id == -1:
                    try:
                        output = api.server(data)          # logits
                    except TypeError:
                        output = api.server.model(data)    # logits
                else:
                    output = api.clients[client_id](data)  # logits

                test_loss += F.cross_entropy(output, target, reduction="sum").item()
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        test_loss /= len(dataloader.dataset)
        accuracy = 100.0 * correct / len(dataloader.dataset)
        print(f"Test set: Average loss: {test_loss:.4f}, Accuracy: {accuracy:.2f}%")

    return _evaluate_global_model


# -------------------------
# 4) Prepare dataloaders (80/20 split, IID split among clients)
# -------------------------
def prepare_dataloader(num_clients: int,
                       myid: int,
                       train: bool = True,
                       path: str = "COVID-19_Radiography_Dataset",
                       seed: int = 0):
    """
    num_clients: total number of clients
    myid: client id in [1..num_clients] (1-based like AIJack tutorial)
    train: True => client train loader, False => global test loader
    path: folder containing class subfolders
    """
    fix_seed(seed)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        # Recommended normalization for RGB images
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = datasets.ImageFolder(root=str(Path(path)), transform=transform)
    num_classes = len(dataset.classes)

    # fixed train/test split (80/20)
    idxs = list(range(len(dataset)))
    random.shuffle(idxs)

    split = int(0.8 * len(idxs))
    train_idxs = idxs[:split]
    test_idxs = idxs[split:]

    if train:
        # split train among clients (IID)
        client_splits = np.array_split(train_idxs, num_clients)
        client_idx = client_splits[myid - 1]  # myid is 1-based
        subset = Subset(dataset, client_idx)

        return DataLoader(
            subset,
            batch_size=training_batch_size,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        ), num_classes
    else:
        subset = Subset(dataset, test_idxs)
        return DataLoader(
            subset,
            batch_size=test_batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        ), num_classes


# -------------------------
# 5) Model (returns logits for CrossEntropy)
# Updated linear layer for IMG_SIZE=128 => feature map 16x16 after 3 pools
# -------------------------
class Net(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64x64
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 32x32
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 16x16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x  # logits


# -------------------------
# 6) Build client loaders + test loader
# -------------------------
fix_seed(seed)

local_dataloaders = []
num_classes = None
for c in range(client_size):
    loader, nc = prepare_dataloader(client_size, myid=c + 1, train=True, seed=seed)
    local_dataloaders.append(loader)
    num_classes = nc

test_dataloader, _ = prepare_dataloader(client_size, myid=1, train=False, seed=seed)

# sanity check batch
x, y = next(iter(local_dataloaders[0]))
print("Batch X:", x.shape, "Batch y:", y.shape)
print("Unique labels in batch:", torch.unique(y))
print("Num classes:", num_classes)


# -------------------------
# 7) AIJack FedAvg setup + run
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

clients = [FedAVGClient(Net(num_classes=num_classes).to(device), user_id=c) for c in range(client_size)]
local_optimizers = [optim.SGD(client.parameters(), lr=lr) for client in clients]

server = FedAVGServer(clients, Net(num_classes=num_classes).to(device))

api = FedAVGAPI(
    server,
    clients,
    criterion,
    local_optimizers,
    local_dataloaders,
    num_communication=num_rounds,
    custom_action=evaluate_global_model(test_dataloader),
)

api.run()


Exists: True
Subfolders: ['Lung_Opacity.metadata.xlsx', 'Viral Pneumonia.metadata.xlsx', 'Viral Pneumonia', 'COVID.metadata.xlsx', 'Normal.metadata.xlsx', 'Lung_Opacity', 'Normal', 'COVID', 'README.md.txt']
Batch X: torch.Size([64, 3, 128, 128]) Batch y: torch.Size([64])
Unique labels in batch: tensor([0, 1, 2, 3])
Num classes: 4
Device: cpu
Test set: Average loss: 1.1874, Accuracy: 48.10%
Test set: Average loss: 1.1623, Accuracy: 48.11%
Test set: Average loss: 1.1346, Accuracy: 48.87%
Test set: Average loss: 1.1005, Accuracy: 51.74%
Test set: Average loss: 1.0613, Accuracy: 53.44%


In [4]:
# =========================
# SECOND RUN: FedAvg with Adam optimizer
# =========================

print("\n\n=== Running FedAvg with Adam optimizer ===\n")

# Reset seed for fair comparison
fix_seed(seed)

# Rebuild clients (fresh models!)
clients_adam = [
    FedAVGClient(Net(num_classes=num_classes).to(device), user_id=c)
    for c in range(client_size)
]

# Adam instead of SGD
local_optimizers_adam = [
    optim.Adam(client.parameters(), lr=lr)
    for client in clients_adam
]

# New server (fresh model)
server_adam = FedAVGServer(
    clients_adam,
    Net(num_classes=num_classes).to(device)
)

# New API instance
api_adam = FedAVGAPI(
    server_adam,
    clients_adam,
    criterion,
    local_optimizers_adam,
    local_dataloaders,
    num_communication=num_rounds,
    custom_action=evaluate_global_model(test_dataloader),
)

api_adam.run()




=== Running FedAvg with Adam optimizer ===

Test set: Average loss: 0.8043, Accuracy: 66.39%
Test set: Average loss: 0.6597, Accuracy: 73.68%
Test set: Average loss: 0.5744, Accuracy: 77.56%
Test set: Average loss: 0.5446, Accuracy: 78.97%
Test set: Average loss: 0.5226, Accuracy: 80.09%


In [8]:
import numpy as np
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms
from pathlib import Path

def make_dirichlet_client_loaders(
    dataset_root="COVID-19_Radiography_Dataset",
    n_clients=5,
    alpha=0.3,
    train=True,
    seed=0,
    batch_size=64,
    img_size=128
):
    fix_seed(seed)

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = datasets.ImageFolder(root=str(Path(dataset_root)), transform=transform)

    # fixed 80/20 split (same as before)
    idxs = np.arange(len(dataset))
    rng = np.random.default_rng(seed)
    rng.shuffle(idxs)
    split = int(0.8 * len(idxs))
    train_idxs = idxs[:split]
    test_idxs  = idxs[split:]

    if not train:
        test_subset = Subset(dataset, test_idxs)
        test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=0)
        return None, test_loader

    # labels for train indices
    y = np.array([dataset.samples[i][1] for i in train_idxs])
    n_classes = len(dataset.classes)

    # collect indices per class
    idx_by_class = [train_idxs[y == c] for c in range(n_classes)]
    for c in range(n_classes):
        rng.shuffle(idx_by_class[c])

    client_indices = [[] for _ in range(n_clients)]

    # Dirichlet allocation per class
    for c in range(n_classes):
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        splits = (np.cumsum(proportions) * len(idx_by_class[c])).astype(int)
        parts = np.split(idx_by_class[c], splits[:-1])
        for k in range(n_clients):
            client_indices[k].extend(parts[k].tolist())

    client_loaders = []
    for k in range(n_clients):
        subset = Subset(dataset, client_indices[k])
        client_loaders.append(DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=0))

    test_subset = Subset(dataset, test_idxs)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=0)

    return client_loaders, test_loader


In [ ]:
print("\n=== RERUN: Non-IID (Dirichlet) FedAvg + Adam ===\n")
fix_seed(seed)

alpha = 0.3  # try 0.3 first; later compare with 1.0
local_dataloaders, test_dataloader = make_dirichlet_client_loaders(
    dataset_root="COVID-19_Radiography_Dataset",
    n_clients=client_size,
    alpha=alpha,
    train=True,
    seed=seed,
    batch_size=training_batch_size,
    img_size=IMG_SIZE
)

# Rebuild clients + server from scratch
clients = [FedAVGClient(Net(num_classes=num_classes).to(device), user_id=c) for c in range(client_size)]
local_optimizers = [optim.Adam(client.parameters(), lr=lr) for client in clients]
server = FedAVGServer(clients, Net(num_classes=num_classes).to(device))

api = FedAVGAPI(
    server,
    clients,
    criterion,
    local_optimizers,
    local_dataloaders,
    num_communication=num_rounds,
    custom_action=evaluate_global_model(test_dataloader),
)
api.run()



=== RERUN: Non-IID (Dirichlet) FedAvg + Adam ===

Test set: Average loss: 0.8859, Accuracy: 64.54%
Test set: Average loss: 0.6937, Accuracy: 70.29%


In [7]:
print("\n=== RERUN: Non-IID (Dirichlet) FedAvg + Adam ===\n")
fix_seed(seed)

alpha = 1.0  # try 0.3 first; later compare with 1.0
local_dataloaders, test_dataloader = make_dirichlet_client_loaders(
    dataset_root="COVID-19_Radiography_Dataset",
    n_clients=client_size,
    alpha=alpha,
    train=True,
    seed=seed,
    batch_size=training_batch_size,
    img_size=IMG_SIZE
)

# Rebuild clients + server from scratch
clients = [FedAVGClient(Net(num_classes=num_classes).to(device), user_id=c) for c in range(client_size)]
local_optimizers = [optim.Adam(client.parameters(), lr=lr) for client in clients]
server = FedAVGServer(clients, Net(num_classes=num_classes).to(device))

api = FedAVGAPI(
    server,
    clients,
    criterion,
    local_optimizers,
    local_dataloaders,
    num_communication=num_rounds,
    custom_action=evaluate_global_model(test_dataloader),
)
api.run()



=== RERUN: Non-IID (Dirichlet) FedAvg + Adam ===

Test set: Average loss: 0.8058, Accuracy: 69.36%
Test set: Average loss: 0.6113, Accuracy: 75.12%
Test set: Average loss: 0.5820, Accuracy: 76.67%
Test set: Average loss: 0.5288, Accuracy: 78.86%
Test set: Average loss: 0.5253, Accuracy: 79.25%
